# Capstone: Custom Gadgets & Neutral-Atom Evaluation

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do you extend a `qodec` with a neutral-atom zone-global logical operation (`hadamard_swap_all`), audit its circuit distance, and benchmark it end-to-end?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Add a new `Instruction` and transversal `Gadget` (`hadamard_swap_all`, `H 0 1 2 3`) to the $C_4$ protocol.
- Synthesize its parity checks with `ec.filled`, verify zero audit issues with `ec.audit`, and confirm $d_{\text{circuit}} = 2$.
- Compile a Q# logical benchmark calling `hadamard_swap_all` and generate a complete evaluation summary.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## Capstone Scenario: Neutral-Atom Zone-Global Operations

You are a QEC researcher and you want to extend our distance-2 `C4` protocol with a **new logical unitary instruction**—**`hadamard_swap_all`**—that exploits a signature hardware capability of a neutral-atom architecture ($^{171}\text{Yb}$ nuclear-spin qubits in optical tweezers):

1. **Zone-Global Optical Raman Pulses ($p_{1\text{Q}} \ll p_{2\text{Q}}$):**
   In a neutral-atom array, all 4 atoms of a $C_4$ code block residing in an optical trap zone can be illuminated simultaneously by a **single zone-global Raman laser pulse**, executing transversal $H^{\otimes 4}$ (`H 0 1 2 3`) in parallel with $>99.9\%$ single-qubit fidelity and **zero two-qubit Rydberg blockade overhead**.
2. **Algebraic Action of a Zone-Global $H^{\otimes 4}$ Pulse on $C_4$:**
   Recall how our $2 \times 2$ $C_4$ block encodes its two logical qubits:
   - $\bar{X}_0 = X_0 X_1 \xrightarrow{H^{\otimes 4}} Z_0 Z_1 = \bar{Z}_1$
   - $\bar{Z}_0 = Z_0 Z_2 \xrightarrow{H^{\otimes 4}} X_0 X_2 = \bar{X}_1$
   - $\bar{X}_1 = X_0 X_2 \xrightarrow{H^{\otimes 4}} Z_0 Z_2 = \bar{Z}_0$
   - $\bar{Z}_1 = Z_0 Z_1 \xrightarrow{H^{\otimes 4}} X_0 X_1 = \bar{X}_0$
   Thus, a zone-global `H 0 1 2 3` pulse implements a valid fault-tolerant logical operation on $C_4$ with Clifford instruction action:
   `qodec.actions.Clifford({"X_0": "Z_1", "Z_0": "X_1", "X_1": "Z_0", "Z_1": "X_0"})`
   (simultaneously applying logical Hadamard $\bar{H}_0 \bar{H}_1$ and swapping logical qubits $0 \leftrightarrow 1$; if an un-swapped logical Hadamard is desired, neutral-atom AOD tweezers can physically swap atoms `1` and `2` via coherent atom shuttling!).
3. **Neutral-Atom Asymmetric Noise Profile:**
   Unlike uniform depolarizing models where 1Q gates, 2Q gates, and measurements share the same error rate $p$, neutral-atom processors exhibit strong physical hierarchy:
   - **1Q Zone-Global Raman Gates (`H`, `X`, `Y`, `Z`):** $p_{1\text{Q}} \approx p / 10$ (e.g. $5 \times 10^{-4}$ at $p = 5 \times 10^{-3}$)
   - **2Q Rydberg Blockade Gates (`CX` / `CZ`):** $p_{2\text{Q}} = p$ (e.g. $5 \times 10^{-3}$, the dominant error source)
   - **Optical Fluorescence Readout/Reset (`mresetz`):** $p_{\text{meas}} \approx p / 2$

## Capstone Workflow Walkthrough

In the capstone cell below, we execute four end-to-end steps:
1. Add the `hadamard_swap_all` instruction and zone-global gadget (`H 0 1 2 3`) to Layer 0 of our upgraded `C4` protocol and synthesize its stabilizer checks with **`ec.filled`**.
2. Verify with **`ec.audit`** and **`ec.GadgetProfile`** that `hadamard_swap_all` is equivalent to its objective and achieves **circuit distance 2**.
3. Compile and run a Q# logical circuit (`TestHadamardSwap`) that prepares $|00\rangle_L$ (`prepare_z_all`), applies the zone-global `hadamard_swap_all` (transforming $|00\rangle_L \to |++\rangle_L$), runs `syndrome`, and measures in the $X$ basis (`measure_x_all`)—comparing uniform depolarizing noise against a **neutral-atom asymmetric noise model**.
4. Generate a structured **QEC Protocol Evaluation Summary**.

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Capstone — Add `hadamard_swap_all`, Audit, Benchmark & Export Report
import json
from _unit import simulation
simulation.ensure_course_state(globals(), stage="upgraded_all")
from qdk import TargetProfile, qsharp
from qdk.simulation import NoiseConfig, run_qir

capstone_protocol = qodec.Qodec.loads(protocol.dumps())
layer0 = capstone_protocol.layers[0]

# 1. Register the `hadamard_swap_all` logical instruction in Layer 0
hswap_instr = qodec.Instruction(
    mnemonic="hadamard_swap_all",
    inputs=[qodec.instructions.BlockOperand("C4")],
    outputs=[qodec.instructions.BlockOperand("C4")],
    action=[qodec.actions.Clifford({"X_0": "Z_1", "Z_0": "X_1", "X_1": "Z_0", "Z_1": "X_0"})],
)
layer0.instruction_set.instructions["hadamard_swap_all"] = hswap_instr

# 2. Add the zone-global `H 0 1 2 3` gadget and synthesize its checks with `ec.filled`
hswap_gadget = qodec.Gadget(
    implements=hswap_instr,
    inputs=list(layer0.gadgets["syndrome"].inputs),
    outputs=list(layer0.gadgets["syndrome"].outputs),
    circuit=qodec.gadgets.Circuit(
        format="stim",
        source="H 0 1 2 3\n",
        instruction_set=capstone_protocol.layers[1].instruction_set,
    ),
)
layer0.gadgets["hadamard_swap_all"] = ec.filled(hswap_gadget)

print("=== Capstone Protocol Audit ===")
audit_report = ec.audit(capstone_protocol)
print(audit_report)
assert len(audit_report.diagnostics) == 0, "Expected zero audit diagnostics!"

# 3. Compile and test a Q# benchmark that uses `hadamard_swap_all`
qsharp.init(target_profile=TargetProfile.Adaptive_RI)
qsharp.eval("""
@Measurement()
operation prepare_z_all(block : Qubit) : Result { body intrinsic; }
@Measurement()
operation measure_x_all(block : Qubit) : (Result, Result) { body intrinsic; }
operation syndrome(block : Qubit) : Unit { body intrinsic; }
operation hadamard_swap_all(block : Qubit) : Unit { body intrinsic; }

operation TestHadamardSwap() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    hadamard_swap_all(block);
    syndrome(block);
    let (x0, x1) = measure_x_all(block);
    ([reject], [x0, x1])
}
""")

hswap_qir = str(qsharp.compile("TestHadamardSwap()"))
hswap_benchmarks = {"HadamardSwap (Z->X)": hswap_qir}
hswap_sweep = simulation.sample_sweep(
    capstone_protocol,
    hswap_benchmarks,
    [1e-3, 5e-3, 1e-2],
    shots=400,
    workers=4,
    max_readout_score=0.05,
)

# Compare Uniform Noise vs. Neutral-Atom Asymmetric Noise at p_2Q = 5e-3
# (where 1Q global Raman pulses have p_1Q = p_2Q / 10 = 5e-4, Rydberg CX has p_2Q = 5e-3, readout has p_meas = 2.5e-3)
na_noise = NoiseConfig()
for gate_1q in ["h", "x", "y", "z"]:
    getattr(na_noise, gate_1q).set_depolarizing(5e-4)
na_noise.cx.set_depolarizing(5e-3)
na_noise.mresetz.x = 2.5e-3

na_decoder = simulation._prepare_postselect_syndrome_decoder
na_rows = run_qir(
    hswap_qir,
    qodec=capstone_protocol,
    decoder=na_decoder,
    noise=na_noise,
    shots=600,
    seed=123,
    type="clifford",
    on_shot_failure="discard",
)
na_accepted = [logical for flags, logical in na_rows if all(b == qsharp.Result.Zero for b in flags)]
na_wrong = sum(any(b != qsharp.Result.Zero for b in logical) for logical in na_accepted)

# 4. Build a structured QEC Protocol Evaluation Summary
gadget_summary = {}
for gname, gobj in layer0.gadgets.items():
    gprof = ec.GadgetProfile(gobj)
    gdist = gprof.distance()
    gadget_summary[gname] = {
        "equivalent_to_objective": gprof.action.is_equivalent_to(gprof.objective),
        "fault_distance": gdist.value,
        "num_checks": len(gobj.checks),
        "num_readouts": len(gobj.readouts),
    }

evaluation_report = {
    "ecosystem": "Neutral-Atom QEC Stack",
    "code": capstone_protocol.layers[0].codes["C4"].name,
    "code_distance": ec.CodeProfile(capstone_protocol.layers[0].codes["C4"]).distance().value,
    "audit_ok": len(audit_report.diagnostics) == 0,
    "gadgets": gadget_summary,
    "uniform_depolarizing_sweep": [
        {
            "p": p,
            "attempted": int(cnt["attempted"]),
            "accepted": int(cnt["accepted"]),
            "wrong": int(cnt["wrong"]),
            "logical_error_rate": round(cnt["wrong"] / cnt["accepted"], 5),
        }
        for p, cnt in hswap_sweep["HadamardSwap (Z->X)"]
    ],
    "neutral_atom_asymmetric_benchmark_at_p2Q_0.005": {
        "p_1Q_raman": 5e-4,
        "p_2Q_rydberg_cx": 5e-3,
        "p_meas": 2.5e-3,
        "attempted": 600,
        "accepted": len(na_accepted),
        "wrong": na_wrong,
        "logical_error_rate": round(na_wrong / len(na_accepted), 5),
    },
}

print("=== QEC Protocol Evaluation Summary ===")
print(json.dumps(evaluation_report, indent=2))

=== Magne Ecosystem Capstone Protocol Audit ===


audit: ok


=== Magne QEC Protocol Evaluation Summary ===
{
  "ecosystem": "Magne Neutral-Atom QEC Stack (Yb-171)",
  "code": "C4",
  "code_distance": 2,
  "audit_ok": true,
  "gadgets": {
    "cx_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 4,
      "num_readouts": 0
    },
    "hadamard_swap_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 2,
      "num_readouts": 0
    },
    "measure_x_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 1,
      "num_readouts": 2
    },
    "measure_z_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 1,
      "num_readouts": 2
    },
    "prepare_x_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 2,
      "num_readouts": 1
    },
    "prepare_z_all": {
      "equivalent_to_objective": true,
      "fault_distance": 2,
      "num_checks": 2,

## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did a zone-global Hadamard pulse (<code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">H 0 1 2 3</code>) on <i>C</i><sub>4</sub> require the Clifford action mapping <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">{'X_0': 'Z_1', 'Z_0': 'X_1', 'X_1': 'Z_0', 'Z_1': 'X_0'}</code> instead of <code style="color:inherit;font-family:var(--vscode-editor-font-family,monospace);background:rgba(0,0,0,0.2);border-radius:3px;padding:0 0.2em;">{'X_0': 'Z_0', 'Z_0': 'X_0', ...}</code>, and how can neutral-atom tweezers undo the logical swap?</summary>

<div style="padding:0.1em 1em;">

Because on $C_4$, $\bar{X}_0 = X_0 X_1$ (top row) and $\bar{Z}_0 = Z_0 Z_2$ (left column), while $\bar{X}_1 = X_0 X_2$ (left column) and $\bar{Z}_1 = Z_0 Z_1$ (top row). Applying $H$ to every physical atom maps $X_0 X_1 \to Z_0 Z_1 = \bar{Z}_1$ and $Z_0 Z_2 \to X_0 X_2 = \bar{X}_1$, simultaneously performing a logical Hadamard and swapping logical qubits 0 and 1. In a neutral-atom processor, physically swapping atoms 1 and 2 via optical tweezer movement (or relabeling) restores the un-swapped logical Hadamard `X_0 → Z_0, X_1 → Z_1` with zero two-qubit Rydberg gates.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Summarize the 5-step QDK-EC engineering loop you used across this course to take a code from definition to benchmarked protocol.</summary>

<div style="padding:0.1em 1em;">

1. **Define & Profile Code:** `qodec.Code` + `ec.CodeProfile(code).distance()`.

</div>

</details></div>

## Exercise: Verify Circuit Distance of Zone-Global hadamard_swap_all

Synthesize the stabilizer parity checks for the neutral-atom zone-global `hadamard_swap_all` gadget (`H 0 1 2 3`) using `ec.filled` and return its static circuit distance.

In [ ]:
from qdk import ec
import qodec
from _unit import exercise, simulation


@exercise
def hadamard_swap_circuit_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    capstone_protocol = qodec.Qodec.loads(ns['protocol'].dumps())
    layer0 = capstone_protocol.layers[0]
    hswap_instr = qodec.Instruction(
        mnemonic='hadamard_swap_all',
        inputs=[qodec.instructions.BlockOperand('C4')],
        outputs=[qodec.instructions.BlockOperand('C4')],
        action=[qodec.actions.Clifford({'X_0': 'Z_1', 'Z_0': 'X_1', 'X_1': 'Z_0', 'Z_1': 'X_0'})],
    )
    hswap_gadget = qodec.Gadget(
        implements=hswap_instr,
        inputs=list(layer0.gadgets['syndrome'].inputs),
        outputs=list(layer0.gadgets['syndrome'].outputs),
        circuit=qodec.gadgets.Circuit(
            format='stim',
            source='H 0 1 2 3\n',
            instruction_set=capstone_protocol.layers[1].instruction_set,
        ),
    )
    # Fill hswap_gadget with ec.filled and return its integer circuit distance
    return ...

**Hint**

Call `filled_g = ec.filled(hswap_gadget)` and return `ec.GadgetProfile(filled_g).distance().value`.

In [ ]:
from qdk import ec
import qodec
from _unit import exercise, simulation


@exercise
def hadamard_swap_circuit_distance() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'upgraded_all')
    capstone_protocol = qodec.Qodec.loads(ns['protocol'].dumps())
    layer0 = capstone_protocol.layers[0]
    hswap_instr = qodec.Instruction(
        mnemonic='hadamard_swap_all',
        inputs=[qodec.instructions.BlockOperand('C4')],
        outputs=[qodec.instructions.BlockOperand('C4')],
        action=[qodec.actions.Clifford({'X_0': 'Z_1', 'Z_0': 'X_1', 'X_1': 'Z_0', 'Z_1': 'X_0'})],
    )
    hswap_gadget = qodec.Gadget(
        implements=hswap_instr,
        inputs=list(layer0.gadgets['syndrome'].inputs),
        outputs=list(layer0.gadgets['syndrome'].outputs),
        circuit=qodec.gadgets.Circuit(
            format='stim',
            source='H 0 1 2 3\n',
            instruction_set=capstone_protocol.layers[1].instruction_set,
        ),
    )
    filled_g = ec.filled(hswap_gadget)
    return ec.GadgetProfile(filled_g).distance().value

Transversal `H 0 1 2 3` acts independently on each physical qubit of the $C_4$ block, so no single physical gate fault can spread to two data qubits, preserving circuit distance `2`.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 10 (Capstone)
- **Capstone Evaluation Rubric:**
  1. **Instruction Specification:** `hadamard_swap_all` Clifford action (`qodec.actions.Clifford({"X_0": "Z_1", "Z_0": "X_1", "X_1": "Z_0", "Z_1": "X_0"})`) accurately reflects the swap between logical qubits 0 and 1 under a zone-global $H^{\otimes 4}$ pulse.
  2. **Automated Check Synthesis & Audit:** `ec.filled` derives the two stabilizer checks swapping $g_X \leftrightarrow g_Z$, and `ec.audit(capstone_protocol)` returns `audit: ok`.
  3. **Circuit Distance Verification:** `ec.GadgetProfile(g).distance().value == 2` across all 7 gadgets.
  4. **End-to-End Q# Execution & Neutral-Atom Noise Comparison:** `TestHadamardSwap` compiles with `TargetProfile.Adaptive_RI` and demonstrates both $O(p^2)$ post-selected logical error suppression and the fidelity advantage of high-fidelity neutral-atom 1Q Raman pulses ($p_{1\text{Q}} = p/10$).
- **Socratic Challenge for Advanced Learners:** Ask: *"In a neutral-atom processor with movable optical tweezers (AODs), how could you modify the `hadamard_swap_all` gadget so it implements pure logical Hadamard (`X_0 → Z_0, Z_0 → X_0, X_1 → Z_1, Z_1 → X_1`) without swapping logical qubits 0 and 1?"*

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../09-simulation-and-decoding/simulation_and_decoding.workbook.ipynb" title="Previous unit: Noisy Simulation, Decoding & Confidence Filtering" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Noisy Simulation, Decoding & Confidence Filtering</span></span></a><span></span></div>